# BirdCLEF+ 2025 - Pretrained models: CORAL diagnostics (no submission)

**CPU only, internet off. This notebook writes `coral_diag.json` only and produces nothing to submit.**
We wrote it because A2 with CORAL heads scored far below the original A2 (0.688 vs 0.848 public), so we first had to rule out a bug in the inference pipeline.

**Part 1, inference path check.** Runs the CORAL inference code on the files of the a2-perch-infer dry run, with (a) the original head and (b) the control head.

**Part 2, window diagnostics.** Applies each head to the raw soundscape embeddings (12000 windows), checks each head's standardiser against the soundscape target, and compares the control heads with the originals.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core

import glob, hashlib, importlib.metadata, json, subprocess, sys, time
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata

T_NOTEBOOK = time.time()
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local test only, never set on Kaggle
ORT_VERSION = "1.31.1"
NUM_WORKERS = int(os.environ.get("M3_INFER_WORKERS", "4"))      # 0 = no Pool (Windows tests)
BUDGET_MIN = float(os.environ.get("M3_DIAG_BUDGET_MIN", "30"))
PART2_RESERVE_MIN = 3.0                                          # allowance for Part 2 before it has been measured
EXPECT_FILES, EXPECT_ROWS, EXPECT_WINDOWS = 20, 240, 12000
PASS_MAX_ABS = 1e-5
INPUT = Path(os.environ.get("M3_DIAG_INPUT", "/kaggle/input"))
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
OUT.mkdir(parents=True, exist_ok=True)

def locate(slug, name):
    """Exactly one file or folder called `name` inside the attached source `slug`."""
    hits = sorted(set(glob.glob(f"{INPUT.as_posix()}/**/{slug}/**/{name}", recursive=True)))
    if len(hits) != 1:
        raise FileNotFoundError(f"expected one {name} in {slug} under {INPUT}, found {len(hits)}: {hits[:5]}")
    return Path(hits[0])

CORAL = "sc4000-birdclef-m3-coral-heads"
A2_EMBED = "sc4000-birdclef-m3-a2-perch-embed"
A2_INFER = "sc4000-birdclef-m3-a2-perch-infer"
A3_MLP = "sc4000-birdclef-m3-a3-mlp-head"
SOUND = "sc4000-birdclef-m3-soundscape-embed"

DATA = locate("birdclef-2025", "sample_submission.csv").parent
ONNX_PATH = locate("sc4000-birdclef-m3-perch-onnx", "perch_v2.onnx")
ORIG_PERCH = locate(A2_EMBED, "perch_head.npz")
ORIG_BIRDMAE = locate(A3_MLP, "birdmae_mlp_head.npz")
REF_CSV = locate(A2_INFER, "submission.csv")          # input only; the original inference on its dry-run files

PERCH_HEADS = {"original": ORIG_PERCH, "meanstd": locate(CORAL, "perch_head_meanstd.npz"),
               "coral_std": locate(CORAL, "perch_head_coral_std.npz"), "control": locate(CORAL, "perch_head_control.npz")}
BIRDMAE_HEADS = {"original": ORIG_BIRDMAE, "control": locate(CORAL, "birdmae_head_control.npz"),
                 "meanstd": locate(CORAL, "birdmae_head_meanstd.npz"), "coral_std": locate(CORAL, "birdmae_head_coral_std.npz")}
PERCH_EMB = locate(SOUND, "perch_soundscape_embeddings.npy")
BIRDMAE_EMB = locate(SOUND, "birdmae_soundscape_embeddings.npy")

# Same install logic as the inference notebooks: exact ONNX Runtime version, loaded first in the workers.
try:
    have = importlib.metadata.version("onnxruntime")
except importlib.metadata.PackageNotFoundError:
    have = None
if have != ORT_VERSION and not SMOKE_TEST:
    wheels = locate(A2_EMBED, "ort_cpu_wheels")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--force-reinstall",
                        "--find-links", str(wheels), f"onnxruntime=={ORT_VERSION}"], capture_output=True, text=True)
    if r.returncode != 0:
        if have is None:
            raise RuntimeError(f"onnxruntime is missing and no wheel fits this Python: {r.stderr[-400:]}")
        print(f"Keeping image onnxruntime {have}; wheel install failed: {r.stderr[-200:]}")
print("onnxruntime image version:", have, "| wanted:", ORT_VERSION)

species = json.load(open(locate(CORAL, "species.json")))
for slug in (A2_EMBED, A3_MLP):
    assert json.load(open(locate(slug, "species.json"))) == species, f"species differ in {slug}"
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"

if not SMOKE_TEST:
    h = hashlib.sha256()
    with open(ONNX_PATH, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    if h.hexdigest() != "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f":
        raise RuntimeError(f"perch_v2.onnx checksum mismatch: {h.hexdigest()}")

results = dict(notes=["No feature transform is applied at inference: features stay raw in the CORAL path, so the identity transform is what this path already does."])
print(results["notes"][0])
def save_results():
    json.dump(results, open(OUT / "coral_diag.json", "w"), indent=1)
def check_budget(projected_min, where):
    if projected_min > BUDGET_MIN:
        save_results()
        raise RuntimeError(f"Timing guard at {where}: projected {projected_min:.1f} min > {BUDGET_MIN:.0f} min")
print("Data:", DATA, "| CPUs:", os.cpu_count(), "| workers:", NUM_WORKERS)

## Worker code (verbatim from the CORAL inference notebook)

In [ ]:
import os, time
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, the fixed Perch 2.0 input
PERCH_SHA256 = "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f"

def to_32k_mono(y, sr):
    """Shared with the other A2 notebook, keep both copies identical."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)     # polyphase, as perch-hoplite does
    return y.astype(np.float32)

def preprocess_windows(x):
    """Perch preprocessing, per window: remove the mean, scale the peak to 0.25.
    Shared with the other A2 notebook, keep both copies identical."""
    x = x - x.mean(axis=1, keepdims=True)
    peak = np.abs(x).max(axis=1, keepdims=True)
    return (x * (0.25 / np.maximum(peak, 1e-8))).astype(np.float32)

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Shared with the other A2 notebook."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

_sess, _head, _init_s = None, None, 0.0

def init_worker(onnx_path, head_path):
    global _sess, _head, _init_s
    t0 = time.perf_counter()
    import onnxruntime as ort          # first load happens inside the worker
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    _sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"])
    assert _sess.get_inputs()[0].name == "inputs"
    _head = dict(np.load(head_path))
    _init_s = time.perf_counter() - t0

def predict_file(path):
    t0 = time.perf_counter()
    y, sr = sf.read(str(path), dtype="float32")
    y = to_32k_mono(y, sr)
    t1 = time.perf_counter()
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    x = preprocess_windows(np.stack([y[i * WIN:(i + 1) * WIN] for i in range(n_win)]))
    t2 = time.perf_counter()
    emb = _sess.run(["embedding"], {"inputs": x})[0]       # the 12 windows of a file are one batch
    t3 = time.perf_counter()
    p = head_predict(_head, emb).astype(np.float32)
    t4 = time.perf_counter()
    # row_id = <file stem>_<end time in seconds>, e.g. soundscape_8358733_5
    row_ids = [f"{path.stem}_{(i + 1) * WIN_SEC}" for i in range(n_win)]
    return row_ids, p, dict(decode=t1 - t0, preprocess=t2 - t1, perch=t3 - t2, head=t4 - t3,
                            init=_init_s, pid=os.getpid())

## Part 1: inference path check

In [ ]:
def run_inference(head_path, test_files):
    """Same Pool and collection code as the CORAL inference notebook, without writing a CSV."""
    t0 = time.time()
    initargs = (ONNX_PATH, head_path)
    if NUM_WORKERS > 0:
        with Pool(NUM_WORKERS, initializer=init_worker, initargs=initargs) as pool:
            res = list(pool.imap(predict_file, test_files, chunksize=1))
    else:
        init_worker(*initargs)
        res = [predict_file(p) for p in test_files]
    row_ids = [r for ids, _, _ in res for r in ids]
    preds = np.concatenate([p for _, p, _ in res])
    stage = {k: float(sum(r[2][k] for r in res)) for k in ("decode", "preprocess", "perch", "head")}
    return row_ids, preds, time.time() - t0, stage

def spearman_per_class(a, b):
    """Spearman per column. Columns constant in a or b are ignored and counted."""
    ra, rb = rankdata(a, axis=0), rankdata(b, axis=0)
    sa, sb = ra.std(axis=0), rb.std(axis=0)
    ok = (sa > 0) & (sb > 0)
    if not ok.any():
        return dict(mean=None, median=None, p5=None, n_valid=0, n_constant=int((~ok).sum()))
    rho = ((ra[:, ok] - ra[:, ok].mean(axis=0)) * (rb[:, ok] - rb[:, ok].mean(axis=0))).mean(axis=0) / (sa[ok] * sb[ok])
    return dict(mean=float(rho.mean()), median=float(np.median(rho)), p5=float(np.percentile(rho, 5)),
                n_valid=int(ok.sum()), n_constant=int((~ok).sum()))

def f3(v):
    return "n/a" if v is None else f"{v:.3f}"

def compare(ref, row_ids, preds):
    """Absolute difference after aligning row_id; ref columns were checked against the species list."""
    assert len(set(row_ids)) == len(row_ids), "duplicate row_id in new predictions"
    assert set(row_ids) == set(ref.index), "row_id sets differ from the reference"
    new = pd.DataFrame(preds, index=row_ids, columns=species).loc[ref.index, species].to_numpy(dtype=np.float64)
    old = ref[species].to_numpy(dtype=np.float64)
    d = np.abs(new - old)
    i, j = np.unravel_index(d.argmax(), d.shape)
    return dict(max_abs=float(d.max()), mean_abs=float(d.mean()), worst_row=str(ref.index[i]), worst_species=species[j]), new, old

ref = pd.read_csv(REF_CSV)
assert list(ref.columns[1:]) == species, "reference submission columns differ from the species list"
assert ref.row_id.is_unique, "duplicate row_id in the reference submission"
assert len(ref) == EXPECT_ROWS, f"expected {EXPECT_ROWS} reference rows, found {len(ref)}"
stems = list(dict.fromkeys(r.rsplit("_", 1)[0] for r in ref.row_id))
assert len(stems) == EXPECT_FILES, f"expected {EXPECT_FILES} files, found {len(stems)}"
test_files = [DATA / "train_soundscapes" / f"{s}.ogg" for s in stems]
missing = [p.name for p in test_files if not p.is_file()]
assert not missing, f"not in train_soundscapes: {missing[:5]}"
ref = ref.set_index("row_id")

# (a) original head: the pure code-path test
t_p1 = time.time()
ids_a, preds_a, wall_a, stage_a = run_inference(ORIG_PERCH, test_files)
cmp_a, _, _ = compare(ref, ids_a, preds_a)
cmp_a["status"] = "PASS" if cmp_a["max_abs"] <= PASS_MAX_ABS else "FAIL"
cmp_a.update(threshold=PASS_MAX_ABS, wall_s=wall_a, stages_s=stage_a)
results["part1a_original_head"] = cmp_a
print(f"Part 1(a) original head: {cmp_a['status']} | max abs {cmp_a['max_abs']:.3e} | mean abs {cmp_a['mean_abs']:.3e} | "
      f"threshold {PASS_MAX_ABS:.0e} | worst {cmp_a['worst_row']} / {cmp_a['worst_species']} | wall {wall_a:.1f}s")
save_results()
check_budget((time.time() - T_NOTEBOOK + wall_a) / 60 + PART2_RESERVE_MIN, "after Part 1(a)")

# (b) control head: a CPU retrain, so not expected to match to 1e-6
ids_b, preds_b, wall_b, stage_b = run_inference(PERCH_HEADS["control"], test_files)
cmp_b, new_b, old_b = compare(ref, ids_b, preds_b)
cmp_b.update(label="retrained head, informational", spearman_per_class=spearman_per_class(new_b, old_b), wall_s=wall_b, stages_s=stage_b)
results["part1b_control_head"] = cmp_b
sb = cmp_b["spearman_per_class"]
print(f"Part 1(b) control head (retrained head, informational): max abs {cmp_b['max_abs']:.3e} | mean abs {cmp_b['mean_abs']:.3e} | "
      f"mean per-class Spearman {f3(sb['mean'])} (valid {sb['n_valid']}, constant {sb['n_constant']}) | wall {wall_b:.1f}s")
T_PART1 = time.time() - t_p1

def stats_diff(head, orig):
    return {k + "_" + m: float(f(head[k] - orig[k])) for k in ("mean", "std")
            for m, f in (("l2", np.linalg.norm), ("max_abs", lambda d: np.abs(d).max()))}

comp = {}
for enc, orig_path, heads in (("perch", ORIG_PERCH, PERCH_HEADS), ("birdmae", ORIG_BIRDMAE, BIRDMAE_HEADS)):
    orig = dict(np.load(orig_path))
    # coral_raw is outside the requested head lists, so it is optional and only compared here.
    extra = glob.glob(f"{INPUT.as_posix()}/**/{CORAL}/**/{enc}_head_coral_raw.npz", recursive=True)
    paths = {k: v for k, v in heads.items() if k != "original"}
    if extra:
        paths["coral_raw"] = extra[0]
    for name, path in paths.items():
        d = stats_diff(dict(np.load(path)), orig)
        comp[f"{enc}/{name}"] = d
        print(f"mean/std vs original | {enc}/{name:<9} | mean L2 {d['mean_l2']:.4f} max {d['mean_max_abs']:.4f} | "
              f"std L2 {d['std_l2']:.4f} max {d['std_max_abs']:.4f}")
results["mean_std_vs_original"] = comp
save_results()
check_budget((time.time() - T_NOTEBOOK) / 60 + PART2_RESERVE_MIN, "after Part 1")

## Part 2: soundscape window diagnostics

In [ ]:
# Rule of thumb for the log only, never raises: head standardiser "matches" the soundscape target.
FS_MEAN_TOL, FS_RATIO_LOW, FS_RATIO_HIGH = 0.10, 0.90, 1.10
END_TO_END_LABEL = "training script end-to-end check, expected about 0.99"

def feature_space(head, emb):
    """Head standardiser against the per-dimension mean/std of the raw soundscape windows."""
    mu_t, sig_t = emb.mean(axis=0), emb.std(axis=0)
    ok = sig_t > 0
    gap = float((np.abs(head["mean"][ok] - mu_t[ok]) / sig_t[ok]).mean())
    ratio = float(np.median(head["std"][ok] / sig_t[ok]))
    return dict(mean_abs_diff_over_target_std=gap, median_std_ratio_head_over_target=ratio, n_zero_std_dims=int((~ok).sum()),
                close_to_target=bool(gap <= FS_MEAN_TOL and FS_RATIO_LOW <= ratio <= FS_RATIO_HIGH))

def window_diag(head, emb, ref_p):
    """Numbers for one head on the raw soundscape embeddings. Returns the row and the probabilities."""
    x = (emb - head["mean"]) / head["std"]
    p = head_predict(head, emb)
    assert np.isfinite(p).all() and p.shape == (len(emb), len(species)), "bad predictions"
    top = p.argmax(axis=1)
    counts = np.bincount(top, minlength=len(species))
    i_top = int(counts.argmax())
    row = dict(mean_prob=float(p.mean()), median_window_max=float(np.median(p.max(axis=1))),
               frac_max_gt_0_5=float((p.max(axis=1) > 0.5).mean()), frac_max_gt_0_9=float((p.max(axis=1) > 0.9).mean()),
               top1_distinct_classes=int((counts > 0).sum()), top1_most_common_share=float(counts[i_top] / len(top)),
               top1_most_common_species=species[i_top],
               std_input_mean_l2=float(np.linalg.norm(x, axis=1).mean()), std_input_mean_abs=float(np.abs(x).mean()))
    row["spearman_vs_original"] = spearman_per_class(p, p if ref_p is None else ref_p)
    return row, p

t_p2 = time.time()
sets = [("perch", "A2 Perch", PERCH_HEADS, PERCH_EMB, ""), ("birdmae", "A3 Bird-MAE", BIRDMAE_HEADS, BIRDMAE_EMB, " (additional)")]
n_heads = sum(len(s[2]) for s in sets)
done, table, part2, fs_table, e2e_table = 0, [], {}, [], []
for enc, label, heads, emb_path, note in sets:
    emb = np.load(emb_path)
    assert emb.shape[0] == EXPECT_WINDOWS, f"{enc}: expected {EXPECT_WINDOWS} windows, found {emb.shape[0]}"
    assert np.isfinite(emb).all(), f"{enc}: non-finite embeddings"
    ref_p, enc_rows, enc_fs = None, {}, {}
    for name, path in heads.items():                      # the original head comes first and is the reference
        head = dict(np.load(path))
        assert head["mean"].shape[0] == emb.shape[1], f"{enc}/{name}: input size differs from the embeddings"
        row, p = window_diag(head, emb, ref_p)
        if ref_p is None:
            ref_p = p
        enc_rows[name] = row
        table.append((f"{label}{note}", name, row))
        if name != "original":
            fs = feature_space(head, emb)
            fs["role"] = "reference (not mapped to target)" if name == "control" else "expected close to target"
            enc_fs[name] = fs
            fs_table.append((enc, name, fs))
        if name == "control":
            e2e_table.append((enc, row["spearman_vs_original"]))
        done += 1
        check_budget((time.time() - T_NOTEBOOK + (time.time() - t_p2) / done * (n_heads - done)) / 60, f"Part 2 after {enc}/{name}")
    part2[enc] = dict(label=label + note, n_windows=int(emb.shape[0]), heads=enc_rows, feature_space=enc_fs)
    del emb, ref_p
results["part2_windows"] = part2
results["part2_note"] = "Mean per-class top-1 share is 1/n_classes by construction, so top-1 concentration is reported as distinct classes and the share of the most common top-1 class."
results["control_vs_original_end_to_end"] = dict(label=END_TO_END_LABEL, spearman_per_class={e: s for e, s in e2e_table})
results["feature_space_rule"] = dict(mean_abs_diff_over_target_std_max=FS_MEAN_TOL, median_std_ratio_range=[FS_RATIO_LOW, FS_RATIO_HIGH], note="log only, never raises")
save_results()
T_PART2 = time.time() - t_p2

print(f"\n{'set':<26} {'head':<10} {'mean p':>8} {'med max':>8} {'>0.5':>6} {'>0.9':>6} {'top1 n':>6} {'top1 sh':>8} {'rho mean':>8} {'rho med':>8} {'rho p5':>8} {'const':>5} {'z L2':>8} {'|z|':>6}")
for label, name, r in table:
    s = r["spearman_vs_original"]
    print(f"{label:<26} {name:<10} {r['mean_prob']:8.4f} {r['median_window_max']:8.4f} {r['frac_max_gt_0_5']:6.3f} {r['frac_max_gt_0_9']:6.3f} "
          f"{r['top1_distinct_classes']:6d} {r['top1_most_common_share']:8.3f} {f3(s['mean']):>8} {f3(s['median']):>8} {f3(s['p5']):>8} {s['n_constant']:5d} "
          f"{r['std_input_mean_l2']:8.2f} {r['std_input_mean_abs']:6.3f}")

print(f"\nControl vs ORIGINAL head, {END_TO_END_LABEL} (per-class Spearman, 12000 windows)")
for enc, s in e2e_table:
    orig_name = "perch_head.npz" if enc == "perch" else "birdmae_mlp_head.npz"
    print(f"{enc}_head_control.npz vs {orig_name}: mean {f3(s['mean'])} | median {f3(s['median'])} | p5 {f3(s['p5'])} | valid {s['n_valid']} | constant {s['n_constant']}")

print(f"\nHead standardiser vs soundscape target (log only; close = mean gap <= {FS_MEAN_TOL} and std ratio in {FS_RATIO_LOW}-{FS_RATIO_HIGH})")
print(f"{'head':<20} {'mean gap / sig_t':>16} {'med std ratio':>14} {'close':>6}  role")
for enc, name, fs in fs_table:
    print(f"{enc + '/' + name:<20} {fs['mean_abs_diff_over_target_std']:16.4f} {fs['median_std_ratio_head_over_target']:14.4f} {str(fs['close_to_target']):>6}  {fs['role']}")
mapped = [fs for _, name, fs in fs_table if name != "control"]
holds = all(fs["close_to_target"] for fs in mapped)
results["feature_space_expectation_holds"] = holds
save_results()
print("Expectation that meanstd/coral_std head mean and std sit close to the soundscape target:", "HOLDS" if holds else "DOES NOT HOLD for at least one head (see table)")

In [ ]:
assert not (OUT / "submission.csv").exists() and not Path("submission.csv").exists(), "this notebook must not write a submission"
total = time.time() - T_NOTEBOOK
results["timing_seconds"] = dict(part1=T_PART1, part2=T_PART2, total=total, budget_min=BUDGET_MIN)
save_results()
print(f"\nPart 1(a) result: {results['part1a_original_head']['status']} | output: coral_diag.json only, no submission file")
print(f"TIMING | setup {total - T_PART1 - T_PART2:.0f}s | part1 {T_PART1:.0f}s | part2 {T_PART2:.0f}s | total {total / 60:.2f} min (guard {BUDGET_MIN:.0f} min)")